# Mão na Roda — Fine-tuning do BERTabaporu
Diagnóstico Preliminar de Falhas Automotivas Baseado em IA

Notebook equivalente ao do BERTimbau, trocando o modelo base para
**BERTabaporu** (`pablocosta/bertabaporu-base-uncased`), pré-treinado em
238 milhões de tweets em português brasileiro (Costa et al., RANLP 2023).

Hipótese sendo testada: por ser pré-treinado em linguagem de rede social
(informal, com gírias e erros ortográficos), o BERTabaporu pode generalizar
melhor nos estilos informal/gíria/erro-ortográfico do dataset do que o
BERTimbau, que foi pré-treinado em texto majoritariamente formal
(Wikipédia + BrWaC).

**Diferença crítica de pré-processamento:** o BERTabaporu é `uncased`
(não diferencia maiúsculas/minúsculas) — diferente do BERTimbau, que é
`cased`. Isso é tratado automaticamente pelo tokenizer do próprio modelo,
mas é importante deixar registrado no TCC como uma diferença de
pré-processamento entre os experimentos, não uma inconsistência.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente de
execução → GPU (T4).

In [1]:
!pip uninstall -y torchvision -q

In [2]:
# ============================================================
# PASSO 1 — Setup e verificação do ambiente (1.1 a 1.3)
# Projeto Mão na Roda — Fine-tuning do BERTabaporu
# ============================================================

# --- 1.1 Instala as dependências ---
!pip install -q transformers datasets accelerate scikit-learn pandas openpyxl

# --- 1.2 Verifica se a GPU está disponível ---
import torch
print("Torch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  ATENÇÃO: nenhuma GPU detectada. Vá em 'Ambiente de execução' > "
          "'Alterar tipo de ambiente de execução' e selecione GPU antes de continuar.")

# --- 1.3 Constantes do projeto ---
MODEL_CHECKPOINT = "pablocosta/bertabaporu-base-uncased"  # BERTabaporu (Costa et al., RANLP 2023)
SEED = 42
NUM_LABELS = 10
MAX_LENGTH = 64

LABEL_NAMES = [
    "Bateria/Sistema Elétrico",
    "Sistema de Freios",
    "Superaquecimento do Motor",
    "Suspensão/Amortecedores",
    "Sistema de Transmissão/Câmbio",
    "Vazamento de Óleo",
    "Sistema de Arrefecimento",
    "Pneu/Roda",
    "Sistema de Injeção/Combustível",
    "Sistema de Escapamento",
]

import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print()
print("Setup concluído. Modelo base:", MODEL_CHECKPOINT)

Torch: 2.11.0+cu128
GPU disponível: True
GPU: Tesla T4

Setup concluído. Modelo base: pablocosta/bertabaporu-base-uncased


## Passo 1 — Setup e verificação do ambiente

In [3]:
# ============================================================
# --- 1.4 Carregando os arquivos de split (via Google Drive) ---
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

CAMINHO_BASE = '/content/drive/MyDrive/Agente Mecânico/notebooks/Finetuning - 3000 relatos/3 Splits - Finetuning/Splits'

import pandas as pd
import os

CONFIGS = ["80_10_10", "70_15_15", "60_20_20"]

splits = {}
for tag in CONFIGS:
    splits[tag] = {
        "treino": pd.read_csv(os.path.join(CAMINHO_BASE, f"split_treino_{tag}.csv")),
        "val":    pd.read_csv(os.path.join(CAMINHO_BASE, f"split_val_{tag}.csv")),
        "teste":  pd.read_csv(os.path.join(CAMINHO_BASE, f"split_teste_{tag}.csv")),
    }
    n_tr, n_val, n_te = len(splits[tag]["treino"]), len(splits[tag]["val"]), len(splits[tag]["teste"])
    print(f"{tag}: treino={n_tr} | val={n_val} | teste={n_te} | total={n_tr+n_val+n_te}")

assert "peso_treino" in splits["80_10_10"]["treino"].columns, \
    "Coluna peso_treino não encontrada — confira se subiu o CSV certo (do v15, não uma versão antiga)."
print("\nColuna peso_treino presente. Pronto para o Passo 2 (tokenização).")

print("Checkpoint definido como:", MODEL_CHECKPOINT)

Mounted at /content/drive
80_10_10: treino=2432 | val=305 | teste=304 | total=3041
70_15_15: treino=2171 | val=435 | teste=435 | total=3041
60_20_20: treino=1824 | val=608 | teste=609 | total=3041

Coluna peso_treino presente. Pronto para o Passo 2 (tokenização).
Checkpoint definido como: pablocosta/bertabaporu-base-uncased


In [4]:
print(MODEL_CHECKPOINT)

pablocosta/bertabaporu-base-uncased


## Passo 2 — Tokenização

O tokenizer é carregado a partir do próprio `MODEL_CHECKPOINT` — isso garante
que o vocabulário e as regras de tokenização usadas batem exatamente com as
que o BERTabaporu usou no seu próprio pré-treino. Usar o tokenizer errado
(ex.: o do BERTimbau num modelo BERTabaporu) quebraria silenciosamente a
qualidade, porque os IDs de token não corresponderiam ao vocabulário que o
modelo aprendeu.

In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_batch(batch):
    return tokenizer(
        batch["relato"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def preparar_dataset_hf(df):
    # Converte um DataFrame pandas em um Dataset do Hugging Face,
    # já tokenizado e com as colunas 'labels' e 'weight' prontas para o Trainer.
    ds = Dataset.from_pandas(df[["relato", "falha_label", "peso_treino"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.rename_column("falha_label", "labels")
    ds = ds.rename_column("peso_treino", "weight")
    ds = ds.remove_columns(["relato"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "weight"])
    return ds

# Tokeniza as 3 configs de uma vez (usadas na comparação leve do Passo 4)
datasets_tokenizados = {}
for tag in CONFIGS:
    datasets_tokenizados[tag] = {
        "treino": preparar_dataset_hf(splits[tag]["treino"]),
        "val":    preparar_dataset_hf(splits[tag]["val"]),
        "teste":  preparar_dataset_hf(splits[tag]["teste"]),
    }
    print(f"{tag}: tokenização concluída "
          f"({len(datasets_tokenizados[tag]['treino'])} treino, "
          f"{len(datasets_tokenizados[tag]['val'])} val, "
          f"{len(datasets_tokenizados[tag]['teste'])} teste)")

# Checagem rápida: mostra um exemplo tokenizado para conferência visual
exemplo = datasets_tokenizados["80_10_10"]["treino"][0]
print("\nExemplo tokenizado:")
print("input_ids (primeiros 15):", exemplo["input_ids"][:15])
print("labels:", exemplo["labels"], "->", LABEL_NAMES[int(exemplo["labels"])])
print("weight:", exemplo["weight"])

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/504k [00:00<?, ?B/s]

Map:   0%|          | 0/2432 [00:00<?, ? examples/s]

Map:   0%|          | 0/305 [00:00<?, ? examples/s]

Map:   0%|          | 0/304 [00:00<?, ? examples/s]

... (5 outputs left out)

Map:   0%|          | 0/1824 [00:00<?, ? examples/s]

Map:   0%|          | 0/608 [00:00<?, ? examples/s]

Map:   0%|          | 0/609 [00:00<?, ? examples/s]

60_20_20: tokenização concluída (1824 treino, 608 val, 609 teste)

Exemplo tokenizado:
input_ids (primeiros 15): tensor([    2,    34,  3715,  1955,  3697,  1932,  2886,  3102,     1,  1973,
         1976,  1963, 10972,  2170, 15415])
labels: tensor(0) -> Bateria/Sistema Elétrico
weight: tensor(0.5000)


## Passo 3 — `WeightedTrainer` e métricas

Idêntico ao usado no notebook do BERTimbau — a lógica de ponderação por
`peso_treino` é independente do modelo, então essa célula não muda entre
experimentos. Isso é proposital: manter o máximo de código idêntico entre
os notebooks de modelos diferentes é o que torna a comparação de resultados
metodologicamente válida (única variável que muda é o `MODEL_CHECKPOINT`).

**Atenção ao `remove_unused_columns=False`** nos argumentos de treino mais
abaixo — sem isso, o `Trainer` descarta a coluna `weight` automaticamente
por achar que é "não usada" pelo modelo, e a ponderação silenciosamente
deixa de funcionar (Shimodaira, 2000, para a fundamentação da ponderação
por importância).

In [6]:
import torch.nn as nn
from transformers import Trainer
from sklearn.metrics import f1_score, accuracy_score, classification_report

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        weights = inputs.pop("weight").float()
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(reduction="none")
        per_example_loss = loss_fct(logits, labels)
        loss = (per_example_loss * weights).sum() / weights.sum()
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }

print("WeightedTrainer e compute_metrics definidos.")

WeightedTrainer e compute_metrics definidos.


## Passo 4 — Treino leve comparativo (3 configs)

Mesmo procedimento do notebook do BERTimbau: poucas épocas, só para
comparar o F1-macro de validação entre as 3 proporções de split e escolher
a vencedora. O conjunto de teste continua intocado até o Passo 6.

In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments

def rodar_treino_leve(tag, epocas=3):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=NUM_LABELS
    )

    args = TrainingArguments(
        output_dir=f"./resultados_leve_bertabaporu_{tag}",
        num_train_epochs=epocas,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        seed=SEED,
        remove_unused_columns=False,   # crítico — não remover
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=datasets_tokenizados[tag]["treino"],
        eval_dataset=datasets_tokenizados[tag]["val"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics_val = trainer.evaluate()
    return metrics_val["eval_f1_macro"], metrics_val["eval_accuracy"]

resultados_leve = {}
for tag in CONFIGS:
    print(f"\n{'='*60}\nTreino leve — config {tag}\n{'='*60}")
    f1, acc = rodar_treino_leve(tag, epocas=3)
    resultados_leve[tag] = {"f1_macro": f1, "accuracy": acc}
    print(f"{tag}: F1-macro (val) = {f1:.4f} | acurácia (val) = {acc:.4f}")

print("\n=== Comparação final (treino leve, BERTabaporu) ===")
for tag, m in sorted(resultados_leve.items(), key=lambda x: -x[1]["f1_macro"]):
    print(f"{tag}: F1-macro={m['f1_macro']:.4f}  acc={m['accuracy']:.4f}")

# Lembrete: diferenças de F1 abaixo de ~0.01 devem ser tratadas como empate
# técnico nesta escala de dataset — escolher pela config com mais dados de
# treino (80/10/10) nesse caso, não cegamente pelo maior número.


Treino leve — config 80_10_10


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: pablocosta/bertabaporu-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	th

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

... (10 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: pablocosta/bertabaporu-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	th

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,1.368569,0.765636,0.754241,0.761513
2,0.309988,0.630691,0.815892,0.817434
3,0.129995,0.657444,0.818403,0.819079


Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.129995,0.657444,3,0.818403,0.819079


60_20_20: F1-macro (val) = 0.8184 | acurácia (val) = 0.8191

=== Comparação final (treino leve, BERTabaporu) ===
70_15_15: F1-macro=0.8976  acc=0.8989
80_10_10: F1-macro=0.8821  acc=0.8820
60_20_20: F1-macro=0.8184  acc=0.8191


## Passo 5 — Escolha da configuração vencedora

Depois de rodar o Passo 4, preencha manualmente qual config venceu (ou foi
escolhida por critério de desempate) antes de seguir para o treino final.

In [8]:
# Ajustem esta linha de acordo com o resultado do Passo 4
CONFIG_VENCEDORA = "80_10_10"  # <-- editar se outra config vencer

print(f"Config escolhida para o treino final: {CONFIG_VENCEDORA}")
print(f"Treino: {len(datasets_tokenizados[CONFIG_VENCEDORA]['treino'])} exemplos")
print(f"Validação: {len(datasets_tokenizados[CONFIG_VENCEDORA]['val'])} exemplos")
print(f"Teste: {len(datasets_tokenizados[CONFIG_VENCEDORA]['teste'])} exemplos (fica intocado até a avaliação final)")

Config escolhida para o treino final: 80_10_10
Treino: 2432 exemplos
Validação: 305 exemplos
Teste: 304 exemplos (fica intocado até a avaliação final)


## Passo 6 — Treino completo com early stopping

Mesma estratégia usada no treino final do BERTimbau: até 15 épocas, com
early stopping (patience=3) monitorando o F1-macro de validação
(Prechelt, 1998). O checkpoint salvo é o de melhor F1 na validação, não
necessariamente o da última época.

In [9]:
from transformers import EarlyStoppingCallback

model_final = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=NUM_LABELS
)

args_final = TrainingArguments(
    output_dir="./resultados_final_bertabaporu",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    seed=SEED,
    remove_unused_columns=False,
    report_to="none",
)

trainer_final = WeightedTrainer(
    model=model_final,
    args=args_final,
    train_dataset=datasets_tokenizados[CONFIG_VENCEDORA]["treino"],
    eval_dataset=datasets_tokenizados[CONFIG_VENCEDORA]["val"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_final.train()

print("\nMelhor F1-macro de validação atingido:",
      trainer_final.state.best_metric)
print("Checkpoint carregado automaticamente (load_best_model_at_end=True).")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: pablocosta/bertabaporu-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	th

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,1.062816,0.516453,0.794354,0.803279
2,0.259125,0.394293,0.872884,0.875410
3,0.102793,0.429832,0.879801,0.881967
4,0.056581,0.547954,0.869351,0.872131
5,0.029723,0.620995,0.880531,0.881967
6,0.016712,0.667030,0.873847,0.875410
7,0.016244,0.700667,0.874470,0.875410
8,0.009240,0.679475,0.877906,0.878689


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Melhor F1-macro de validação atingido: 0.8805312861229411
Checkpoint carregado automaticamente (load_best_model_at_end=True).


## Passo 7 — Avaliação final no conjunto de teste

Só agora o conjunto de teste é usado — pela primeira e única vez, com o
checkpoint de melhor validação já carregado.

In [11]:
resultado_teste = trainer_final.evaluate(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
print("=== Resultado final no TESTE (BERTabaporu) ===")
print(f"F1-macro: {resultado_teste['eval_f1_macro']:.4f}")
print(f"Acurácia: {resultado_teste['eval_accuracy']:.4f}")

# Relatório por classe — essencial para comparar com o BERTimbau,
# em especial no par Superaquecimento x Arrefecimento
preds_output = trainer_final.predict(datasets_tokenizados[CONFIG_VENCEDORA]["teste"])
preds = np.argmax(preds_output.predictions, axis=-1)
labels_reais = preds_output.label_ids

print("\n=== Relatório por classe ===")
print(classification_report(labels_reais, preds, target_names=LABEL_NAMES, digits=3))

Training Loss,Validation Loss,Epoch,F1 Macro,Accuracy
0.009240,0.665557,8,0.861728,0.865132


=== Resultado final no TESTE (BERTabaporu) ===
F1-macro: 0.8617
Acurácia: 0.8651



=== Relatório por classe ===
                                precision    recall  f1-score   support

      Bateria/Sistema Elétrico      0.966     0.903     0.933        31
             Sistema de Freios      0.879     0.935     0.906        31
     Superaquecimento do Motor      0.762     0.533     0.627        30
       Suspensão/Amortecedores      0.778     0.903     0.836        31
 Sistema de Transmissão/Câmbio      0.939     1.000     0.969        31
             Vazamento de Óleo      0.931     0.900     0.915        30
      Sistema de Arrefecimento      0.758     0.833     0.794        30
                     Pneu/Roda      0.963     0.867     0.912        30
Sistema de Injeção/Combustível      0.781     0.833     0.806        30
        Sistema de Escapamento      0.903     0.933     0.918        30

                      accuracy                          0.865       304
                     macro avg      0.866     0.864     0.862       304
                  weighted avg  

## Passo 8 — Salvar o modelo

Mesma recomendação de sempre: salvar direto no Google Drive, não só no
disco efêmero do Colab (risco já identificado com o BERTimbau).

In [12]:
from google.colab import drive
drive.mount('/content/drive')

CAMINHO_SALVAR = '/content/drive/MyDrive/Agente Mecânico/modelos/bertabaporu_mao_na_roda/'

trainer_final.save_model(CAMINHO_SALVAR)
tokenizer.save_pretrained(CAMINHO_SALVAR)

print(f"Modelo e tokenizer salvos em: {CAMINHO_SALVAR}")
print("Conteúdo salvo:")
!ls -la "{CAMINHO_SALVAR}"

import json
resumo = {
    "modelo": MODEL_CHECKPOINT,
    "config_split": CONFIG_VENCEDORA,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "f1_macro_teste": float(resultado_teste["eval_f1_macro"]),
    "acuracia_teste": float(resultado_teste["eval_accuracy"]),
    "melhor_f1_validacao": float(trainer_final.state.best_metric),
}
with open(CAMINHO_SALVAR + "resumo_treino.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=2)

print("\nResumo salvo em resumo_treino.json:")
print(json.dumps(resumo, ensure_ascii=False, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo e tokenizer salvos em: /content/drive/MyDrive/Agente Mecânico/modelos/bertabaporu_mao_na_roda/
Conteúdo salvo:
total 529645
-rw------- 1 root root      1110 Sep  9 04:16 config.json
-rw------- 1 root root 540827680 Sep  9 04:16 model.safetensors
-rw------- 1 root root       379 Sep  9 04:16 tokenizer_config.json
-rw------- 1 root root   1520262 Sep  9 04:16 tokenizer.json
-rw------- 1 root root      5165 Sep  9 04:16 training_args.bin

Resumo salvo em resumo_treino.json:
{
  "modelo": "pablocosta/bertabaporu-base-uncased",
  "config_split": "80_10_10",
  "seed": 42,
  "max_length": 64,
  "f1_macro_teste": 0.8617275341729664,
  "acuracia_teste": 0.8651315789473685,
  "melhor_f1_validacao": 0.8805312861229411
}
